# Crisis Negotiator — GRPO v2 Training (Kaggle 2×T4)

All v2 improvements: multi-turn trajectory scoring, all difficulties, stronger diversity penalty, Q-network guidance, ToM reward.

| | |
|---|---|
| **Hardware** | Kaggle 2×T4 (16 GB each) |
| **Precision** | fp16 |
| **Prompts** | 128 (reduced from 256 for T4 time budget) |
| **Generations** | 4 (reduced from 8 for T4 VRAM) |
| **Epochs** | 1 |
| **Multi-turn** | 3-step trajectory scoring |

**Settings → Accelerator → GPU T4 ×2** before running.

In [ ]:
# Cell 1 — Install dependencies
!pip install -q "openenv-core>=0.2.3" "trl>=0.14" peft accelerate datasets sentence-transformers matplotlib
print('Dependencies installed.')

In [ ]:
# Cell 2 — Clone repo
import os, pathlib, subprocess, shutil

REPO = 'https://github.com/Dinesh052/Test.git'
BRANCH = 'kaggle_version'

if not pathlib.Path('server').exists():
    subprocess.run(['git', 'clone', '-b', BRANCH, REPO, '_repo'], check=True)
    for item in pathlib.Path('_repo').iterdir():
        dest = pathlib.Path(item.name)
        if not dest.exists():
            item.rename(dest)
    shutil.rmtree('_repo', ignore_errors=True)
    print(f'Cloned {BRANCH} branch.')
else:
    print('Already inside repo.')

print('Key files:', [f for f in sorted(os.listdir('.')) if f.endswith('.py')][:10])

In [ ]:
# Cell 3 — GPU check
import torch
n = torch.cuda.device_count()
for i in range(n):
    props = torch.cuda.get_device_properties(i)
    print(f'GPU {i}: {props.name} | {props.total_mem / 1024**3:.1f} GB')
print(f'Total GPUs: {n}')
assert n >= 1, 'No GPU!'

In [ ]:
# Cell 4 — Smoke-test environment
import sys
sys.path.insert(0, '.')
from server.environment import CrisisNegotiatorEnvironment
from models import NegotiatorAction

env = CrisisNegotiatorEnvironment()
obs = env.reset(task_id='easy_domestic_desperate', seed=42)
print(f'Reset OK | phase={obs.phase}')
step_obs = env.step(NegotiatorAction(action_type='emotional_label',
    content="It sounds like you're feeling overwhelmed.",
    reasoning='empathy', target='hostage_taker'))
print(f'Step OK  | reward={step_obs.reward:.4f} done={step_obs.done}')
print('Environment smoke-test PASSED ✓')

In [ ]:
# Cell 5 — Patch train_local_v2.py for Kaggle T4
import pathlib

src = pathlib.Path('train_local_v2.py').read_text()

# 1. Reduce prompts: 256 → 128 (T4 time budget ~30 min)
src = src.replace('num_episodes: int = 256', 'num_episodes: int = 128')

# 2. Reduce generations: 8 → 4 (T4 VRAM)
src = src.replace('num_generations: int = 8', 'num_generations: int = 4')

# 3. Reduce epochs: 2 → 1 (time)
src = src.replace('num_epochs: int = 2', 'num_epochs: int = 1')

# 4. Reduce multi-turn: 4 → 3 (speed — less deepcopy)
src = src.replace('multi_turn_steps: int = 4', 'multi_turn_steps: int = 3')

# 5. bf16 → fp16 (T4 doesn't support bf16 well)
src = src.replace('bf16=True,', 'bf16=False,\n        fp16=True,')
src = src.replace('torch_dtype=torch.bfloat16,', 'torch_dtype=torch.float16,')

# 6. Add generation_batch_size (required by some TRL versions)
src = src.replace(
    'num_generations=CFG.num_generations,\n        max_completion_length=CFG.max_new_tokens,',
    'num_generations=CFG.num_generations,\n        generation_batch_size=CFG.num_generations,\n        max_completion_length=CFG.max_new_tokens,'
)

pathlib.Path('train_local_v2.py').write_text(src)

print('Patched train_local_v2.py for Kaggle T4:')
print('  ✓ num_episodes: 256 → 128')
print('  ✓ num_generations: 8 → 4')
print('  ✓ num_epochs: 2 → 1')
print('  ✓ multi_turn_steps: 4 → 3')
print('  ✓ bf16 → fp16')
print('  ✓ generation_batch_size added')

# Verify
for check in ['num_episodes: int = 128', 'num_generations: int = 4', 'fp16=True']:
    assert check in pathlib.Path('train_local_v2.py').read_text(), f'FAIL: {check}'
print('\nAll patches verified ✓')

In [ ]:
# Cell 5b — Train Q-network first (used by GRPO v2 for action guidance)
import time
t0 = time.time()
%run train_q_network.py --episodes 1000
print(f'\nQ-network trained in {(time.time()-t0)/60:.1f} min')

In [ ]:
# Cell 6 — Train!
import os, sys, time
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'
sys.path.insert(0, '.')

t0 = time.time()
%run train_local_v2.py
elapsed = (time.time() - t0) / 60
print(f'\nTraining finished in {elapsed:.1f} min')

In [ ]:
# Cell 7 — Plot training curve
import json, numpy as np, matplotlib.pyplot as plt

with open('crisis_training_log_v2.json') as f:
    log = json.load(f)

steps = [e['global_step'] for e in log]
rewards = [e['reward'] for e in log]
diffs = [e.get('difficulty', '?') for e in log]

window = min(32, len(rewards))
rolling = np.convolve(rewards, np.ones(window)/window, mode='valid')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Crisis Negotiator — GRPO v2 Training')

# Left: overall
DIFF_COLORS = {'easy': '#55A868', 'medium': '#DD8452', 'hard': '#C44E52'}
colors = [DIFF_COLORS.get(d, '#999') for d in diffs]
ax1.scatter(steps, rewards, c=colors, alpha=0.3, s=12)
ax1.plot(range(window-1, len(rewards)), rolling, 'k-', lw=2, label=f'rolling mean w={window}')
ax1.axhline(0.755, color='grey', ls='--', lw=1, label='random 0.755')
ax1.axhline(0.950, color='green', ls='--', lw=1, label='heuristic 0.950')
ax1.set_xlabel('Step'); ax1.set_ylabel('Reward'); ax1.legend(fontsize=7)
ax1.set_ylim(-0.55, 1.05)

# Right: by difficulty
for diff, color in DIFF_COLORS.items():
    dr = [r for r, d in zip(rewards, diffs) if d == diff]
    if dr:
        w = min(16, len(dr))
        ax2.plot(np.convolve(dr, np.ones(w)/w, mode='valid'), color=color, label=f'{diff} (n={len(dr)})')
ax2.set_xlabel('Steps within difficulty'); ax2.set_ylabel('Rolling Reward')
ax2.legend(fontsize=8)

plt.tight_layout()
plt.savefig('reward_curve_training_v2.png', dpi=150)
plt.show()

print(f'Steps: {len(log)}')
print(f'Mean reward (all): {np.mean(rewards):.4f}')
print(f'Mean reward (last 32): {np.mean(rewards[-32:]):.4f}')

In [ ]:
# Cell 8 — Evaluate trained model
from eval_baselines import TrainedPolicy, run_episodes, summarize, RandomPolicy, HeuristicPolicy
import json

print('=== Random ===')
random_res = run_episodes(RandomPolicy(seed=0), 30, ['easy','medium','hard'], seed_offset=10000)

print('\n=== Heuristic ===')
heur_res = run_episodes(HeuristicPolicy(), 30, ['easy','medium','hard'], seed_offset=10000)

print('\n=== Trained v2 ===')
trained = TrainedPolicy('./crisis-negotiator-trained-v2', 'Qwen/Qwen2.5-3B-Instruct')
trained_res = run_episodes(trained, 30, ['easy','medium','hard'], seed_offset=10000)

results = {'random': summarize(random_res), 'heuristic': summarize(heur_res), 'trained_v2': summarize(trained_res)}
print('\n' + json.dumps(results, indent=2))

import pathlib
pathlib.Path('eval_summary_v2.json').write_text(json.dumps(results, indent=2))
pathlib.Path('eval_trained_v2.json').write_text(json.dumps(trained_res, indent=2))
print('Saved eval_summary_v2.json + eval_trained_v2.json')

In [ ]:
# Cell 9 — Save artifacts
import shutil, pathlib

artifacts = ['crisis-negotiator-trained-v2', 'crisis_training_log_v2.json',
             'reward_curve_training_v2.png', 'eval_summary_v2.json', 'eval_trained_v2.json',
             'q_network.pt', 'q_network_log.json']
for a in artifacts:
    p = pathlib.Path(a)
    if p.exists():
        print(f'  ✓ {a} ({p.stat().st_size/1024:.0f} KB)' if p.is_file() else f'  ✓ {a}/ (dir)')
    else:
        print(f'  ✗ {a}')

if pathlib.Path('crisis-negotiator-trained-v2').exists():
    shutil.make_archive('crisis-negotiator-trained-v2', 'zip', '.', 'crisis-negotiator-trained-v2')
    print('\n📦 crisis-negotiator-trained-v2.zip ready for download')